# Лабораторная работа № 2
## Геометрические преобразования изображений: аффинные и проективные преобразования

**Цель:** изучить масштабирование, перенос, поворот, аффинное преобразование по трём соответствиям и исправление перспективы по четырём точкам.

Используются OpenCV, NumPy и Matplotlib. Исходники `photo.png` и `plane.png` должны лежать в этой папке рядом с ноутбуком. Результаты сохраняются в каталог `results/`, исходники не изменяются. Выбор четырёх углов сетки выполняется щелчками мыши в порядке: левый верхний, правый верхний, правый нижний, левый нижний. Для `plt.ginput` нужен интерактивный backend Matplotlib; если ноутбук работает с inline-картинками, включите `%matplotlib widget` (пакет `ipympl`) либо используйте вариант с вводом координат вручную в соответствующей ячейке.

### 1. Импорты и пути к данным
Если файлы находятся в другом месте, измените `DATA_DIR`. Размер изображения OpenCV имеет форму `(высота, ширина, каналы)`, а функции геометрических преобразований принимают выходной размер `(ширина, высота)`. 

In [12]:
from pathlib import Path
import cv2 as cv
import numpy as np
import matplotlib
matplotlib.use('TkAgg')
import matplotlib.pyplot as plt

DATA_DIR = Path('.')  # папка с photo.png и plane.png
OUT_DIR = DATA_DIR / 'results'
OUT_DIR.mkdir(parents=True, exist_ok=True)

photo_path = DATA_DIR / 'photo.png'
plane_path = DATA_DIR / 'plane.png'
photo = cv.imread(str(photo_path), cv.IMREAD_COLOR)
plane = cv.imread(str(plane_path), cv.IMREAD_COLOR)
if photo is None:
    raise FileNotFoundError(f'Не удалось открыть {photo_path.resolve()}. Поместите photo.png в папку lab2 или измените DATA_DIR.')
if plane is None:
    raise FileNotFoundError(f'Не удалось открыть {plane_path.resolve()}. Поместите plane.png в папку lab2 или измените DATA_DIR.')

height, width = photo.shape[:2]
plane_h, plane_w = plane.shape[:2]
print(f'photo.png: ширина={width}, высота={height}, shape={photo.shape}, dtype={photo.dtype}')
print(f'plane.png: ширина={plane_w}, высота={plane_h}, shape={plane.shape}, dtype={plane.dtype}')

photo.png: ширина=1280, высота=720, shape=(720, 1280, 3), dtype=uint8
plane.png: ширина=558, высота=563, shape=(563, 558, 3), dtype=uint8


В OpenCV цвет хранится как BGR; перед выводом через Matplotlib преобразуем в RGB. Определим небольшую функцию отображения, чтобы не путать порядок каналов.

In [13]:
def show_bgr(image, title='', ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 5))
    ax.imshow(cv.cvtColor(image, cv.COLOR_BGR2RGB))
    ax.set_title(title)
    ax.axis('off')
    return ax

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
show_bgr(photo, 'photo.png — исходная фотография', axes[0])
show_bgr(plane, 'plane.png — плоскость судоку под углом', axes[1])
plt.tight_layout()
plt.show()

## 2. Масштабирование
Уменьшим ширину и высоту вдвое с `INTER_AREA`, увеличим вдвое с `INTER_LINEAR`. Увеличение создаёт интерполированные пиксели, но не восстанавливает детали, отсутствующие в исходнике.

In [14]:
small = cv.resize(photo, None, fx=0.5, fy=0.5, interpolation=cv.INTER_AREA)
large = cv.resize(photo, None, fx=2.0, fy=2.0, interpolation=cv.INTER_LINEAR)
print('Исходный размер (W×H):', (width, height))
print('Уменьшенный размер (W×H):', (small.shape[1], small.shape[0]))
print('Увеличенный размер (W×H):', (large.shape[1], large.shape[0]))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
show_bgr(photo, f'Исходное {width}×{height}', axes[0])
show_bgr(small, f'Уменьшение 0.5×: {small.shape[1]}×{small.shape[0]}', axes[1])
show_bgr(large, f'Увеличение 2×: {large.shape[1]}×{large.shape[0]}', axes[2])
plt.tight_layout()
plt.show()

assert small.shape[:2] == (height // 2, width // 2), 'Проверьте округление размеров уменьшенного изображения'
assert large.shape[:2] == (height * 2, width * 2)

Исходный размер (W×H): (1280, 720)
Уменьшенный размер (W×H): (640, 360)
Увеличенный размер (W×H): (2560, 1440)


In [15]:
for filename, result in [('photo_small.png', small), ('photo_large.png', large)]:
    ok = cv.imwrite(str(OUT_DIR / filename), result)
    if not ok:
        raise IOError(f'Не удалось сохранить {filename}')
print('Сохранены:', OUT_DIR / 'photo_small.png', 'и', OUT_DIR / 'photo_large.png')

Сохранены: results/photo_small.png и results/photo_large.png


## 3. Перенос на 100 пикселей вправо и 50 вниз
Матрица переноса оставляет исходный холст `(width, height)`. Поэтому слева и сверху появляются чёрные области, а части снимка, вышедшие за правый и нижний края, обрезаются.

In [16]:
translation_matrix = np.float32([[1, 0, 100], [0, 1, 50]])
shifted = cv.warpAffine(photo, translation_matrix, (width, height))
print('Матрица переноса:\n', translation_matrix)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
show_bgr(photo, 'Исходное изображение', axes[0])
show_bgr(shifted, 'Сдвиг: dx=+100, dy=+50', axes[1])
plt.tight_layout(); plt.show()
cv.imwrite(str(OUT_DIR / 'photo_shifted.png'), shifted)

Матрица переноса:
 [[  1.   0. 100.]
 [  0.   1.  50.]]


True

## 4. Поворот на 30° против часовой стрелки
Поворот выполняется относительно центра. Масштаб равен 1, размер холста остаётся исходным, поэтому угловые области могут стать чёрными, а выступающие части фотографии обрезаются.

In [17]:
center = ((width - 1) / 2.0, (height - 1) / 2.0)
rotation_matrix = cv.getRotationMatrix2D(center, 30, 1.0)
rotated = cv.warpAffine(photo, rotation_matrix, (width, height))
print('Центр:', center)
print('Матрица поворота:\n', rotation_matrix)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
show_bgr(photo, 'Исходное изображение', axes[0])
show_bgr(rotated, 'Поворот +30° (против часовой стрелки)', axes[1])
plt.tight_layout(); plt.show()
cv.imwrite(str(OUT_DIR / 'photo_rotated_30deg.png'), rotated)

Центр: (639.5, 359.5)
Матрица поворота:
 [[  0.8660254    0.5        -94.07324572]
 [ -0.5          0.8660254  367.91386734]]


True

## 5. Аффинное преобразование по трём парам точек
Зададим точки по долям ширины `w` и высоты `h` из условия. Порядок точек в массивах одинаковый: A, B, C. Три исходные точки не лежат на одной прямой.

In [18]:
src_points = np.float32([[0.2 * width, 0.2 * height],
                        [0.8 * width, 0.2 * height],
                        [0.2 * width, 0.8 * height]])
dst_points = np.float32([[0.1 * width, 0.2 * height],
                        [0.8 * width, 0.1 * height],
                        [0.3 * width, 0.9 * height]])
affine_matrix = cv.getAffineTransform(src_points, dst_points)
affine = cv.warpAffine(photo, affine_matrix, (width, height))
print('Исходные точки A, B, C:\n', src_points)
print('Новые положения A′, B′, C′:\n', dst_points)
print('Аффинная матрица 2×3:\n', affine_matrix)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
show_bgr(photo, 'Исходник и исходные точки', axes[0])
axes[0].scatter(src_points[:, 0], src_points[:, 1], c=['red','lime','cyan'], s=60)
for label, (x, y) in zip(['A','B','C'], src_points): axes[0].text(x, y, f' {label}', color='white', fontsize=12)
show_bgr(affine, 'Аффинное преобразование', axes[1])
axes[1].scatter(dst_points[:, 0], dst_points[:, 1], c=['red','lime','cyan'], s=60)
for label, (x, y) in zip(['A′','B′','C′'], dst_points): axes[1].text(x, y, f' {label}', color='white', fontsize=12)
plt.tight_layout(); plt.show()
cv.imwrite(str(OUT_DIR / 'photo_affine.png'), affine)

Исходные точки A, B, C:
 [[ 256.  144.]
 [1024.  144.]
 [ 256.  576.]]
Новые положения A′, B′, C′:
 [[ 128.  144.]
 [1024.   72.]
 [ 384.  648.]]
Аффинная матрица 2×3:
 [[ 1.16666667e+00  5.92592593e-01 -2.56000000e+02]
 [-9.37500000e-02  1.16666667e+00  0.00000000e+00]]


True

Проверим преобразование точки A подстановкой в матрицу: `[x′, y′]ᵀ = M·[x, y, 1]ᵀ`. Вычисленные координаты должны совпасть с заданным положением A′ с малой погрешностью округления.

Аффинное преобразование может менять наклон и масштабы по направлениям, но сохраняет прямолинейность и параллельность прямых. Поэтому его недостаточно для устранения перспективного схождения линий плоского объекта.

In [19]:
A_h = np.array([src_points[0, 0], src_points[0, 1], 1.0])
A_calculated = affine_matrix @ A_h
print('A после умножения матрицы:', A_calculated)
print('Заданное A′:', dst_points[0])
print('Разность:', A_calculated - dst_points[0])
assert np.allclose(A_calculated, dst_points[0], atol=1e-3)

A после умножения матрицы: [128. 144.]
Заданное A′: [128. 144.]
Разность: [-5.68434189e-14  0.00000000e+00]


## 6. Выбор углов судоку и исправление перспективы
Сначала отметим на исходном `plane.png` внешние углы именно сетки, а не листа вокруг неё. Нужно четыре угла в порядке обхода периметра: **левый верхний → правый верхний → правый нижний → левый нижний**. Не переходите по диагонали.

Для работы щелчками нужен интерактивный backend Matplotlib. В JupyterLab обычно можно выполнить `%matplotlib widget` (при установленном `ipympl`) или `%matplotlib qt`. После выполнения кода щёлкните четыре точки; результат выведется на графике.

In [20]:
# При необходимости раскомментируйте ОДНУ строку и перезапустите kernel:
%matplotlib tk

fig, ax = plt.subplots(figsize=(12, 8))
show_bgr(plane, 'Щёлкните углы сетки: ЛВ, ПВ, ПН, ЛН', ax)
plt.tight_layout()
points_clicked = plt.ginput(4, timeout=0, show_clicks=True)
plt.close(fig)
if len(points_clicked) != 4:
    raise ValueError(f'Ожидалось 4 точки, получено {len(points_clicked)}. Запустите ячейку снова.')
src_quad = np.float32(points_clicked)
print('Выбранные точки (x, y), в порядке ЛВ, ПВ, ПН, ЛН:\n', src_quad)

Выбранные точки (x, y), в порядке ЛВ, ПВ, ПН, ЛН:
 [[ 73.95503   85.729744]
 [487.55698   70.68967 ]
 [515.3811   517.37976 ]
 [ 35.602848 512.1158  ]]


Если используемый backend не поддерживает `ginput`, выполните следующую ячейку, посмотрите на отображённое изображение с координатной сеткой и вручную впишите четыре пары `(x, y)` в том же порядке. После ручного ввода пропустите ячейку выше или замените значение `src_quad`.

In [21]:
# Резервный способ: включите эту ячейку и заполните реальные координаты углов.
# Сначала запустите её для просмотра сетки, затем впишите значения и повторно выполните.
fig, ax = plt.subplots(figsize=(12, 8))
show_bgr(plane, 'plane.png — координаты x вправо, y вниз', ax)
ax.set_xticks(np.arange(0, plane_w, max(50, plane_w // 12)))
ax.set_yticks(np.arange(0, plane_h, max(50, plane_h // 12)))
ax.grid(color='yellow', alpha=0.35)
plt.show()

# Пример формы массива (НЕ координаты для вашего снимка):
# src_quad = np.float32([[x_ЛВ, y_ЛВ], [x_ПВ, y_ПВ], [x_ПН, y_ПН], [x_ЛН, y_ЛН]])

Зададим квадрат результата размером 500 × 500. Координаты крайних пикселей равны 0 и 499, поэтому вершины в массиве назначения заканчиваются значением 499, а размер для `warpPerspective` задаётся как `(500, 500)`. Гомография связывает четыре исходные точки с четырьмя вершинами квадрата.

In [22]:
dst_quad = np.float32([[0, 0], [499, 0], [499, 499], [0, 499]])
if 'src_quad' not in globals():
    raise NameError('Сначала выберите четыре угла: выполните ячейку с ginput или задайте src_quad вручную.')
if src_quad.shape != (4, 2):
    raise ValueError(f'src_quad должен иметь форму (4, 2), сейчас {src_quad.shape}')
homography = cv.getPerspectiveTransform(src_quad, dst_quad)
rectified = cv.warpPerspective(plane, homography, (500, 500))
print('Матрица гомографии 3×3:\n', homography)
print('Размер результата:', rectified.shape[1], '×', rectified.shape[0])
fig, axes = plt.subplots(1, 2, figsize=(13, 6))
show_bgr(plane, 'Исходная фотография и углы сетки', axes[0])
axes[0].scatter(src_quad[:, 0], src_quad[:, 1], c='yellow', s=55)
for label, (x, y) in zip(['ЛВ','ПВ','ПН','ЛН'], src_quad): axes[0].text(x, y, label, color='yellow', fontsize=11)
show_bgr(rectified, 'Исправленная перспектива: 500×500', axes[1])
plt.tight_layout(); plt.show()
cv.imwrite(str(OUT_DIR / 'sudoku_rectified_500x500.png'), rectified)
print('Сохранено:', OUT_DIR / 'sudoku_rectified_500x500.png')

Матрица гомографии 3×3:
 [[ 1.32099577e+00  1.18819728e-01 -1.07880670e+02]
 [ 5.15286714e-02  1.41703785e+00 -1.25293097e+02]
 [ 1.30087947e-04  3.94910847e-04  1.00000000e+00]]
Размер результата: 500 × 500
Сохранено: results/sudoku_rectified_500x500.png


## 7. Сводка результатов и вывод
В каталоге `results/` должны появиться уменьшенная и увеличенная версии, перенесённое и повёрнутое изображения, результат аффинного преобразования и выровненная сетка. Сверьте их с исходниками и числами размеров. Если результат перспективного выравнивания выглядит перекрученным, проверьте порядок четырёх углов: обе последовательности должны идти по периметру в одном направлении.

**Вывод.** Масштабирование меняет число пикселей, а интерполяция определяет, как вычисляются новые значения. Перенос и поворот описываются аффинными матрицами; при сохранении холста возможны чёрные области и обрезание. Аффинное преобразование по трём парам точек способно менять наклон и растяжение, сохраняя параллельность прямых. Для плоскости, снятой под углом, требуется проективное преобразование по четырём парам точек: оно компенсирует перспективное схождение и отображает выбранную сетку в квадрат. Точность результата зависит от выбора внешних углов; изгиб листа и оптические искажения одной гомографией не устраняются.